This notebook serves for purposes of anonymisation of the private data set and further calculation of the disclosure risk

0. Load data

In [86]:
import pandas as pd
from pathlib import Path
# Path to data folder
DATA_DIR = Path("../data/H")
orig_df = pd.read_excel(DATA_DIR / "private_dataH.xlsx")
public_register = pd.read_excel(DATA_DIR / "public_data_registerH.xlsx")
copy_df= orig_df.copy()
copy_df.shape

(200, 9)

1. Identifying quasi-identifiers, checking amount of unique people

In [87]:
print(copy_df.columns) #checking features and their types
print(copy_df.dtypes)

not_overlapping_features=[]
for column in copy_df.columns:                #don't need to worry that much for non-overlapping features
    if column not in public_register:
        not_overlapping_features.append(column)
print(not_overlapping_features)

qs_id=["sex", "dob", "zip", "citizenship", "marital_status"] # quasi-identifiers: every feature beside non-overlapping features and sensitive info
group_sizes=copy_df.groupby(qs_id).size()
group_sizes=group_sizes.sort_values()
print("We have " + str((group_sizes == 1).sum()) + " unique entries") #How many unique people, all 200

qs_id_no_dob=["sex", "zip", "citizenship", "marital_status"]   #How many unique people if we don't include dob
group_sizes=copy_df.groupby(qs_id_no_dob).size()
group_sizes=group_sizes.sort_values()
print("We have " + str((group_sizes == 1).sum()) + " unique entries")
    
#25 unique people without dob identifiers. Thus, we need to generalize dob first

Index(['name', 'sex', 'evote', 'dob', 'zip', 'education', 'citizenship',
       'marital_status', 'party'],
      dtype='object')
name                      object
sex                       object
evote                      int64
dob               datetime64[ns]
zip                        int64
education                 object
citizenship               object
marital_status            object
party                     object
dtype: object
['evote', 'education', 'party']
We have 200 unique entries
We have 25 unique entries


2. Generalization and supression of quasi-identifiers

In [88]:
copy_df=orig_df.copy()
copy_df=copy_df.drop(columns=["name"]) # Obviously we don't include direct identifiers

In [89]:

reference_date=pd.Timestamp("2025-07-01")  #Date of conducting the interview
copy_df["age"]= reference_date.year - copy_df["dob"].dt.year
copy_df["age_group"] = pd.cut(
    copy_df["age"],
    bins=[0, 29, 39, 49, 59, 120],
    labels=["0-29", "30-39", "40-49", "50-59", "60+"]
)
copy_df.drop(columns=["dob", "age"], inplace=True)
copy_df.head()

,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,Male,1,2400,Primary education,Denmark,Married/separated,Green,60+
1,Female,0,2400,Vocational Education and Training (VET),Denmark,Never married,Green,30-39
2,Female,1,2100,Masters programmes,Denmark,Never married,Red,30-39
3,Female,0,2100,Primary education,Denmark,Married/separated,Green,60+
4,Female,0,2300,Vocational bachelors educations,Denmark,Married/separated,Red,50-59


In [90]:
qi = ["sex", "age_group", "zip", "citizenship", "marital_status"]

group_sizes = copy_df.groupby(qi).size()

print((group_sizes == 1).sum())   # Checking whether grouping by age helped. Now we have 56 unique people indead of 200

56


C:\Users\shere\AppData\Local\Temp\ipykernel_4964\2148992798.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sizes = copy_df.groupby(qi).size()


Generalization by age helped a lot. Let's look at citizenship quasi-identifier

In [91]:
group_sizes=copy_df.groupby("citizenship").size()
print(group_sizes)  # We discover many people have unique citizenship

copy_df["citizenship"] = copy_df["citizenship"].apply(lambda x:"Denmark" if x == "Denmark" else "Other")
copy_df.head()

citizenship
Bosnia and Herzegovina      1
Denmark                   180
India                       1
Iraq                        1
Japan                       1
Kosovo                      1
Morocco                     1
Nigeria                     1
Norway                      1
Poland                      1
Romania                     1
Stateless                   2
Syria                       2
Thailand                    1
Turkey                      2
USA                         1
United Kingdom              1
Yugoslavia                  1
dtype: int64


,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,Male,1,2400,Primary education,Denmark,Married/separated,Green,60+
1,Female,0,2400,Vocational Education and Training (VET),Denmark,Never married,Green,30-39
2,Female,1,2100,Masters programmes,Denmark,Never married,Red,30-39
3,Female,0,2100,Primary education,Denmark,Married/separated,Green,60+
4,Female,0,2300,Vocational bachelors educations,Denmark,Married/separated,Red,50-59


In [92]:
group_sizes = copy_df.groupby(qi).size()

print((group_sizes == 1).sum()) # Checking whether grouping by citizenship helped. Now we have 52 unique people indead of 56

52


C:\Users\shere\AppData\Local\Temp\ipykernel_4964\879379720.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sizes = copy_df.groupby(qi).size()


I want to check which QI affects the most uniqness

In [93]:
qi = ["sex", "age_group", "zip", "citizenship", "marital_status"]
unique_people_counter = {}

for identifier in qi:
    qi_test = [x for x in qi if x != identifier]
    group_sizes = copy_df.groupby(qi_test).size()
    unique_people_counter[identifier] = (group_sizes == 1).sum()

print(unique_people_counter) #seems that age group still contributes the most to the amount of unique people, but also ZIP code, marital status

{'sex': 29, 'age_group': 12, 'zip': 14, 'citizenship': 33, 'marital_status': 16}


C:\Users\shere\AppData\Local\Temp\ipykernel_4964\3827296790.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sizes = copy_df.groupby(qi_test).size()
C:\Users\shere\AppData\Local\Temp\ipykernel_4964\3827296790.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sizes = copy_df.groupby(qi_test).size()
C:\Users\shere\AppData\Local\Temp\ipykernel_4964\3827296790.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning

Generalizing ZIP

In [94]:
copy_df["zip"].unique()

copy_df["zip"] = copy_df["zip"].map({
    2100: "Area A",
    2200: "Area A",
    2300: "Area B",
    2400: "Area B"
})
group_sizes = copy_df.groupby(qi).size()

print((group_sizes == 1).sum()) # Now we have only 22 unique people

22


C:\Users\shere\AppData\Local\Temp\ipykernel_4964\3223919653.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sizes = copy_df.groupby(qi).size()


Now turn of marital status

In [95]:
copy_df["marital_status"].value_counts()

copy_df["marital_status"] = copy_df["marital_status"].replace({
    "Married/separated": "Ever married",
    "Divorced": "Ever married",
    "Widowed": "Ever married"
})

group_sizes = copy_df.groupby(qi).size()

print((group_sizes == 1).sum()) # Decreased amount of unique people to 13

13


C:\Users\shere\AppData\Local\Temp\ipykernel_4964\3798564472.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sizes = copy_df.groupby(qi).size()


We still have 13 unique people, lets take a look at them

In [96]:
group_sizes.value_counts().sort_index()

print(group_sizes[group_sizes == 1])  #Checking remaining unique people. Notice that it is mostly people with "Other" citizenship

(copy_df["citizenship"] == "Other").sum()

sex     age_group  zip     citizenship  marital_status
Female  0-29       Area B  Denmark      Ever married      1
        30-39      Area B  Other        Ever married      1
                                        Never married     1
        40-49      Area B  Other        Ever married      1
        50-59      Area A  Other        Ever married      1
                   Area B  Other        Ever married      1
        60+        Area B  Other        Ever married      1
Male    0-29       Area B  Other        Never married     1
        30-39      Area A  Other        Ever married      1
        40-49      Area B  Other        Ever married      1
        50-59      Area A  Denmark      Ever married      1
                           Other        Never married     1
                   Area B  Other        Ever married      1
dtype: int64


20

In [97]:
qi_test = [x for x in qi if x != "citizenship"]

group_sizes = copy_df.groupby(qi_test, observed=True).size()

print((group_sizes == 1).sum()) # Supressing citizenship column leaves only 3 unique people.


3


Getting into account the fact that we have only 20 people with "Other" citizenship ,and 11 out of 13
unique remaining people have "Other" citizenship - it is reasonable to supress citizenship column and remaining 3 unique entries after. It will lead only to 1.5 % loss of data: 1 - 197/200 = 0.015

In [98]:
anon_df = copy_df.drop(columns=["citizenship"]).copy()

In [ ]:
qi = ["sex", "age_group", "zip", "marital_status"]
group_sizes = anon_df.groupby(qi, observed=True).transform("size")


0      16
1       5
2       8
3       8
4      12
       ..
195     4
196     9
197    10
198     6
199    19
Name: party, Length: 200, dtype: int64

In [103]:
anon_df = anon_df.loc[group_sizes >= 2].copy()

In [106]:
print(anon_df.shape)
anon_df.head()

(197, 7)


,sex,evote,zip,education,marital_status,party,age_group
0,Male,1,Area B,Primary education,Ever married,Green,60+
1,Female,0,Area B,Vocational Education and Training (VET),Never married,Green,30-39
2,Female,1,Area A,Masters programmes,Never married,Red,30-39
3,Female,0,Area A,Primary education,Ever married,Green,60+
4,Female,0,Area B,Vocational bachelors educations,Ever married,Red,50-59


3.Evaluating anonymisation

3.1Calculating k-anonymity 

In [113]:
k_anonymity = anon_df.groupby(qi, observed=True).size().min()
print("K-anonymity = " + str(k_anonymity))

K-anonymity = 2


3.2 Calculating l-diversity

In [120]:
min_l= anon_df.groupby(qi, observed=True)["party"].nunique().min()
avg_l= anon_df.groupby(qi, observed =True)["party"].nunique().mean()
print("Minimum l score: "+ str(min_l) + ". Average l score: " + str(avg_l))

Minimum l score: 1. Average l score: 1.875


In [ ]:
counts_of_unique_parties=anon_df.groupby(qi, observed=True)["party"].transform("nunique")
(counts_of_unique_parties == 1).sum() # 30 people with attribute disclosure risk

30

3.3 Calculating average risk of reidentification

In [ ]:
eval_df =anon_df.copy()
group_sizes= eval_df.groupby(qi, observed=True).transform("size")
eval_df["Risk"] = 1/ group_sizes
print(eval_df["Risk"].mean())  #Average risk of reidentification is 16 %

0.16243654822335024


Defining the anonymisation function

In [125]:

import pandas as pd

def anonymise_data(file_path):
    df = pd.read_excel(file_path).copy()

    # Remove direct identifiers and citizenship
    df.drop(columns=["name", "citizenship"], inplace=True)

    # Calculate exact age on survey date
    reference_date = pd.Timestamp("2025-07-01")
    dob = pd.to_datetime(df["dob"])

    age = (
        reference_date.year - dob.dt.year

        ).astype(int)
    

    # Generalize age
    df["age_group"] = pd.cut(
        age,
        bins=[0, 29, 39, 49, 59, 120],
        labels=["0-29", "30-39", "40-49", "50-59", "60+"],
        include_lowest=True
    )

    df.drop(columns=["dob"], inplace=True)

    # Generalize ZIP
    df["zip"] = df["zip"].map({
        2100: "Area A",
        2200: "Area A",
        2300: "Area B",
        2400: "Area B"
    })

    # Generalize marital status
    df["marital_status"] = df["marital_status"].replace({
        "Married/separated": "Ever married",
        "Divorced": "Ever married",
        "Widowed": "Ever married"
    })

    # Suppress unique records
    qi = ["sex", "age_group", "zip", "marital_status"]

    group_sizes = df.groupby(qi, observed=True).transform("size")
    df = df.loc[group_sizes >= 2].copy()

    return df


Defining evaluation function

In [126]:

def evaluate_anonymity(df):
    qi = ["sex", "age_group", "zip", "marital_status"]

    # K-anonymity
    groups = df.groupby(qi, observed=True)
    group_sizes = groups.size()
    person_sizes = groups.transform("size")

    # L-diversity
    l_scores = groups["party"].nunique()
    person_l_scores = groups["party"].transform("nunique")

    # Re-identification risk
    risk = 1 / person_sizes

    metrics = {
        "Respondents": len(df),
        "K-anonymity": group_sizes.min(),
        "Minimum l-diversity": l_scores.min(),
        "Average l-diversity": l_scores.mean(),
        "Groups with l=1": (l_scores == 1).sum(),
        "People in l=1 groups": (person_l_scores == 1).sum(),
        "Average sample risk": risk.mean(),
        "Maximum sample risk": risk.max()
    }

    return metrics


Checking that functions output the same values as before from manual operations

In [129]:
output= anonymise_data(DATA_DIR / "private_dataH.xlsx")
result = evaluate_anonymity(output)
print(result)

{'Respondents': 197, 'K-anonymity': 2, 'Minimum l-diversity': 1, 'Average l-diversity': 1.875, 'Groups with l=1': 6, 'People in l=1 groups': 30, 'Average sample risk': 0.16243654822335024, 'Maximum sample risk': 0.5}
